In [1]:
# Installing snowflake connector
import sys
!{sys.executable} -m pip install snowflake-connector-python

INFO: pip is looking at multiple versions of pyopenssl to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   ------ --------------------------------- 1.8/12.1 MB 8.4 MB/s eta 0:00:02
   ---------- ----------------------------- 3.1/12.1 MB 8.8 MB/s eta 0:00:02
   ------------ --------------------------- 3.7/12.1 MB 7.8 MB/s eta 0:00:02
   ------------ --------------------------- 3.7/12.1 MB 7.8 MB/s eta 0:00:02
   ------------ --------------------------- 3.7/12.1 MB 7.8 MB/s eta 0:00:02
   ------------ --------------------------- 3.7/12.1 MB 7.8 MB/s eta 0:00:02
   ------------ --------------------------- 3.7/12.1 MB 7.8 MB/s eta 0:00:02
   ------------ --------------------------- 3.7/12.1 MB 7.8 MB/s eta 0:00:02
   ------------ --------------------------- 3.7/12.1 MB 7.8 MB/s eta 0:00:02
   ------------ --------------------------- 3.7/12.1 MB 7.8 MB/s eta 0:00:02
   -----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.12.3 requires botocore<1.34.70,>=1.34.41, but you have botocore 1.42.63 which is incompatible.


In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

# Set seed so our data is reproducible
np.random.seed(42)
random.seed(42)

In [17]:
# ── MASTER CONFIGURATION ──────────────────────────────────────────

# 8 Global Regions
REGIONS = [
    'India - South Asia',
    'USA - North America', 
    'Germany - Europe',
    'China - Asia Pacific',
    'Brazil - Latin America',
    'UAE - Middle East',
    'Australia - Oceania',
    'South Africa - Africa'
]

# 12 Product Categories
CATEGORIES = [
    'Engine Components',
    'Electrical Systems',
    'Transmission Parts',
    'Brake Systems',
    'Cooling Systems',
    'Hydraulic Parts',
    'Filters & Fluids',
    'Body & Frame',
    'Suspension Parts',
    'Exhaust Systems',
    'Fuel Systems',
    'Safety Equipment'
]

# 15 Suppliers
SUPPLIERS = [
    'Bosch Industrial', 'Siemens Parts Co', 'Denso Supply Chain',
    'Valeo Components', 'ZF Friedrichshafen', 'Magna International',
    'Continental AG', 'Aptiv Solutions', 'BorgWarner Inc',
    'Delphi Technologies', 'Tenneco Auto', 'Mahle GmbH',
    'Schaeffler Group', 'SKF Bearings', 'NSK Industries'
]

# ABC Classification distribution
ABC_DIST = {'A': 0.20, 'B': 0.30, 'C': 0.50}

# XYZ Classification distribution  
XYZ_DIST = {'X': 0.30, 'Y': 0.40, 'Z': 0.30}

print(f" Master config ready!")
print(f"   Regions: {len(REGIONS)}")
print(f"   Categories: {len(CATEGORIES)}")
print(f"   Suppliers: {len(SUPPLIERS)}")

 Master config ready!
   Regions: 8
   Categories: 12
   Suppliers: 15


In [19]:
# ── GENERATING DIM_SKU TABLE ─────────────────────────────────────────

print("Generating 50,000 SKUs... please wait...")

n_skus = 50000

# Generate SKU IDs
sku_ids = [f"SKU-{str(i).zfill(6)}" for i in range(1, n_skus + 1)]

# Assign ABC class
abc_classes = np.random.choice(
    list(ABC_DIST.keys()), 
    size=n_skus, 
    p=list(ABC_DIST.values())
)

# Assign XYZ class
xyz_classes = np.random.choice(
    list(XYZ_DIST.keys()), 
    size=n_skus, 
    p=list(XYZ_DIST.values())
)

# Unit cost based on ABC class
unit_costs = []
for abc in abc_classes:
    if abc == 'A':
        unit_costs.append(round(np.random.uniform(500, 5000), 2))
    elif abc == 'B':
        unit_costs.append(round(np.random.uniform(100, 499), 2))
    else:
        unit_costs.append(round(np.random.uniform(10, 99), 2))

# Lead time in days based on supplier reliability
lead_times = np.random.randint(7, 45, size=n_skus)

# Safety stock multiplier
safety_stock_days = np.random.randint(7, 30, size=n_skus)

dim_sku = pd.DataFrame({
    'SKU_ID': sku_ids,
    'SKU_NAME': [f"Service Part {sku}" for sku in sku_ids],
    'CATEGORY': np.random.choice(CATEGORIES, size=n_skus),
    'SUPPLIER': np.random.choice(SUPPLIERS, size=n_skus),
    'ABC_CLASS': abc_classes,
    'XYZ_CLASS': xyz_classes,
    'COMBINED_CLASS': [f"{a}{x}" for a, x in zip(abc_classes, xyz_classes)],
    'UNIT_COST_USD': unit_costs,
    'LEAD_TIME_DAYS': lead_times,
    'SAFETY_STOCK_DAYS': safety_stock_days,
    'IS_ACTIVE': np.random.choice([True, False], size=n_skus, p=[0.85, 0.15]),
    'CREATED_DATE': pd.Timestamp('2022-01-01')
})

print(f" DIM_SKU generated: {dim_sku.shape[0]:,} rows x {dim_sku.shape[1]} columns")
print(f"\nABC Distribution:")
print(dim_sku['ABC_CLASS'].value_counts())
print(f"\nXYZ Distribution:")
print(dim_sku['XYZ_CLASS'].value_counts())
dim_sku.head(3)

Generating 50,000 SKUs... please wait...
 DIM_SKU generated: 50,000 rows x 12 columns

ABC Distribution:
ABC_CLASS
C    24907
B    15063
A    10030
Name: count, dtype: int64

XYZ Distribution:
XYZ_CLASS
Y    19919
X    15080
Z    15001
Name: count, dtype: int64


,SKU_ID,SKU_NAME,CATEGORY,SUPPLIER,ABC_CLASS,XYZ_CLASS,COMBINED_CLASS,UNIT_COST_USD,LEAD_TIME_DAYS,SAFETY_STOCK_DAYS,IS_ACTIVE,CREATED_DATE
0,SKU-000001,Service Part SKU-000001,Body & Frame,BorgWarner Inc,A,Y,AY,3449.48,29,29,False,2022-01-01
1,SKU-000002,Service Part SKU-000002,Cooling Systems,Magna International,C,Y,CY,93.66,40,22,True,2022-01-01
2,SKU-000003,Service Part SKU-000003,Exhaust Systems,Schaeffler Group,C,X,CX,42.88,33,17,True,2022-01-01


In [ ]:
# ── GENERATE FACT_DEMAND TABLE ────────────────────────────────────
# We'll generate monthly demand per SKU per Region = 50,000 x 8 x 36 months


print("Generating FACT_DEMAND table...")
print("This may take 2-3 minutes - please wait... ☕")

records = []
months = pd.date_range(start='2022-01-01', end='2024-12-31', freq='MS')

# Use only active SKUs
active_skus = dim_sku[dim_sku['IS_ACTIVE'] == True].copy()

for _, sku_row in active_skus.iterrows():
    sku_id = sku_row['SKU_ID']
    abc = sku_row['ABC_CLASS']
    xyz = sku_row['XYZ_CLASS']
    
    # Base demand depends on ABC class
    if abc == 'A':
        base_demand = np.random.randint(100, 500)
    elif abc == 'B':
        base_demand = np.random.randint(20, 100)
    else:
        base_demand = np.random.randint(1, 20)
    
    # Variability depends on XYZ class
    if xyz == 'X':
        noise_factor = 0.10   # Low variability
    elif xyz == 'Y':
        noise_factor = 0.30   # Medium variability
    else:
        noise_factor = 0.70   # High variability

    for region in REGIONS:
        # Regional demand multiplier
        region_mult = np.random.uniform(0.6, 1.4)
        
        for month in months:
            # Seasonality factor
            season = 1 + 0.2 * np.sin(2 * np.pi * month.month / 12)
            
            # Trend factor (slight growth over time)
            trend = 1 + 0.02 * ((month.year - 2022) * 12 + month.month) / 36
            
            # Noise
            noise = np.random.normal(1, noise_factor)
            noise = max(0.1, noise)  # no negative demand
            
            # Final demand
            demand = int(base_demand * region_mult * season * trend * noise)
            demand = max(0, demand)
            
            # Actual vs forecast (forecast has some error)
            forecast_error = np.random.normal(1, 0.15)
            forecast = int(demand * forecast_error)
            forecast = max(0, forecast)
            
            records.append({
                'DEMAND_DATE': month,
                'SKU_ID': sku_id,
                'REGION': region,
                'ACTUAL_DEMAND': demand,
                'FORECAST_DEMAND': forecast,
                'VARIANCE': demand - forecast,
                'UNIT_COST_USD': sku_row['UNIT_COST_USD'],
                'DEMAND_VALUE_USD': demand * sku_row['UNIT_COST_USD'],
                'ABC_CLASS': abc,
                'XYZ_CLASS': xyz,
                'SUPPLIER': sku_row['SUPPLIER'],
                'CATEGORY': sku_row['CATEGORY']
            })

fact_demand = pd.DataFrame(records)
print(f"\n FACT_DEMAND generated!")
print(f"   Rows: {fact_demand.shape[0]:,}")
print(f"   Columns: {fact_demand.shape[1]}")
print(f"   Total Demand Value: ${fact_demand['DEMAND_VALUE_USD'].sum():,.0f}")
fact_demand.head(3)

Generating FACT_DEMAND table...
This may take 2-3 minutes - please wait... ☕


In [15]:
# ── SAVE TO CSV ───────────────────────────────────────────────────

import os

# Creating a folder to save files
os.makedirs('supply_chain_data', exist_ok=True)

print("Saving files... please wait...")

# Saving each table
dim_sku.to_csv('supply_chain_data/DIM_SKU.csv', index=False)
print(f" DIM_SKU.csv saved — {dim_sku.shape[0]:,} rows")

dim_date.to_csv('supply_chain_data/DIM_DATE.csv', index=False)
print(f" DIM_DATE.csv saved — {dim_date.shape[0]:,} rows")

fact_demand.to_csv('supply_chain_data/FACT_DEMAND.csv', index=False)
print(f" FACT_DEMAND.csv saved — {fact_demand.shape[0]:,} rows")

print(f"\n All files saved in: supply_chain_data/ folder")
print(f"   Location: {os.path.abspath('supply_chain_data')}")

Saving files... please wait...
✅ DIM_SKU.csv saved — 50,000 rows
✅ DIM_DATE.csv saved — 1,096 rows
✅ FACT_DEMAND.csv saved — 12,244,320 rows

📁 All files saved in: supply_chain_data/ folder
   Location: C:\Users\Debojyoti (DJ)\supply_chain_data


In [13]:
# ── REGENERATE DIM_DATE ───────────────────────────────────────────

date_range = pd.date_range(start='2022-01-01', end='2024-12-31', freq='D')

dim_date = pd.DataFrame({
    'DATE_KEY': date_range,
    'YEAR': date_range.year,
    'QUARTER': date_range.quarter,
    'MONTH': date_range.month,
    'MONTH_NAME': date_range.strftime('%B'),
    'WEEK': date_range.isocalendar().week.astype(int),
    'DAY_OF_WEEK': date_range.dayofweek,
    'DAY_NAME': date_range.strftime('%A'),
    'IS_WEEKEND': date_range.dayofweek >= 5,
    'IS_MONTH_END': date_range.is_month_end,
    'IS_QUARTER_END': date_range.is_quarter_end,
    'FISCAL_YEAR': date_range.year,
    'FISCAL_QUARTER': date_range.quarter
})

print(f"✅ DIM_DATE generated: {dim_date.shape[0]:,} rows")

✅ DIM_DATE generated: 1,096 rows
